In [1]:
import os

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv

load_dotenv()

print("[OK] OpenAI API 키 설정됨" if os.getenv("OPENAI_API_KEY") else "[ERROR] OpenAI API 키 미설정")
print("[OK] Anthropic API 키 설정됨" if os.getenv("ANTHROPIC_API_KEY") else "[ERROR] Anthropic API 키 미설정")
print("[OK] Google API 키 설정됨" if os.getenv("GOOGLE_API_KEY") else "[ERROR] Google API 키 미설정")
print("[OK] LangSmith 키 설정됨" if os.getenv("LANGSMITH_API_KEY") else "[ERROR] LangSmith 키 미설정")

print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

[OK] OpenAI API 키 설정됨
[ERROR] Anthropic API 키 미설정
[OK] Google API 키 설정됨
[OK] LangSmith 키 설정됨
LangChain: 1.4.1
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("{fruit}의 색상은 무엇입니까?")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 직렬화 가능 여부 확인
print(f"ChatOpenAI: {llm.is_lc_serializable()}")

ChatOpenAI: True


In [6]:
# 체인 직렬화 : 모든 객체를 딕셔너리 또는 JSON 문자열로 변환하는 과정

chain = prompt | llm

chain.is_lc_serializable()

True

In [ ]:
from langchain_core.load import dumpd, dumps

# dumpd : 딕셔너리 형태로 변환
dumpd_chain = dumpd(chain)
print(type(dumpd_chain))

# dumps : Json 형태로 변환
dumps_chain = dumps(chain)
print(type(dumps_chain))

<class 'dict'>
<class 'str'>


In [ ]:
import pickle

# dict을 Pick 파일로 저장 wb="write binary"
with open("fruit_chain.pkl", "wb") as f:
    pickle.dump(dumpd_chain, f)

In [ ]:
import json

# dict을 Json 파일로 저장
with open("fruit_chain.json", "w", encoding="utf-8") as f:
    json.dump(dumpd_chain, f, ensure_ascii=False, indent=2) # indent 정렬

In [20]:
# pkl 파일을 읽음
with open("fruit_chain.pkl", "rb") as f:
    # python 객체로 복원
    loaded_chain = pickle.load(f)
    
print(type(loaded_chain))

<class 'dict'>


In [ ]:
from langchain_core.load import load

# dict 를 다시 chain 객체로 복원, 외부 서비스 연결된 객체가 포함된 경우 allowed 설정 필요(신뢰 할 수 있는 데이터만)
# Chain 객체 복원 + ChatOpenAI 복원 허용
chain_from_file = load(loaded_chain, allowed_objects="all")

print(type(chain_from_file))

# 체인을 실행합니다.
response = chain_from_file.invoke({"fruit": "사과"})

print(response.content)

<class 'langchain_core.runnables.base.RunnableSequence'>
사과의 색상은 다양합니다. 일반적으로 빨간색, 초록색, 노란색 등이 있으며, 일부 사과는 이들 색상이 혼합된 형태로도 나타납니다. 예를 들어, 레드 딜리셔스 사과는 진한 빨간색을 띠고, 그라니 스미스 사과는 선명한 초록색을 가지고 있습니다. 또한, 골든 딜리셔스 사과는 노란색을 띠고 있습니다. 사과의 색상은 품종에 따라 다르며, 익은 정도에 따라서도 변화할 수 있습니다.


In [37]:
# Chain 객체 복원 + ChatOpenAI 복원 허용 + API Key 연결
load_chain = load(loaded_chain,
    secrets_map=
    {
        "OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]
    },
    allowed_objects="all"
)

response = load_chain.invoke({"fruit": "사과"})

print(response.content)

사과의 색상은 다양합니다. 일반적으로 빨간색, 초록색, 노란색 등 여러 가지 색상이 있으며, 품종에 따라 다르게 나타납니다. 예를 들어, 레드 딜리셔스는 주로 빨간색이고, 그라니 스미스는 초록색이며, 골든 딜리셔스는 노란색입니다. 또한, 일부 사과는 두 가지 이상의 색상이 섞여 있을 수도 있습니다.


In [36]:
with open("fruit_chain.json", "r", encoding="utf-8") as fp:
    loaded_from_json_chain = json.load(fp)
    
loads_chain = load(loaded_from_json_chain, allowed_objects="all")

response = loads_chain.invoke({"fruit": "사과"})

print(response.content)

사과의 색상은 다양합니다. 일반적으로 빨간색, 초록색, 노란색 등이 있으며, 일부 사과는 이들 색상이 혼합된 형태로도 나타납니다. 예를 들어, 레드 딜리셔스 사과는 진한 빨간색을 띠고, 그라니 스미스 사과는 선명한 초록색을 가지고 있습니다. 또한, 일부 사과는 갈색이나 보라색을 띠기도 합니다. 사과의 색상은 품종에 따라 다르며, 익은 정도에 따라서도 변화할 수 있습니다.
